In [ ]:
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
import boto3
from botocore.exceptions import ClientError
from loguru import logger

# Add project path
path = r"C:\Users\Admin\Documents\V03-120126"
sys.path.append(path)

from constants import MinioConfig


# ===== INIT MINIO CLIENT =====
def init_minio_client():
    try:
        client = boto3.client(
            "s3",
            endpoint_url=MinioConfig.ENDPOINT,
            aws_access_key_id=MinioConfig.ACCESS_KEY,
            aws_secret_access_key=MinioConfig.SECRET_KEY,
        )
        logger.info("MinIO client initialized")
        return client
    except Exception as e:
        logger.error(f"Init MinIO failed: {e}")
        raise


# ===== ENSURE BUCKET EXISTS =====
def ensure_bucket(client, bucket_name):
    try:
        client.head_bucket(Bucket=bucket_name)
        logger.info(f"Bucket exists: {bucket_name}")
    except ClientError:
        logger.info(f"Creating bucket: {bucket_name}")
        client.create_bucket(Bucket=bucket_name)


# ===== LIST OBJECTS =====
def list_all_objects(client, bucket_name):
    paginator = client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket_name):
        for obj in page.get("Contents", []):
            yield obj["Key"]


# ===== COPY OBJECT (NO PREFIX) =====
def copy_object(
    client,
    source_bucket,
    object_key,
    target_bucket,
):
    try:
        client.copy_object(
            Bucket=target_bucket,
            Key=object_key,   # 🔥 giữ nguyên key
            CopySource={
                "Bucket": source_bucket,
                "Key": object_key
            },
        )
    except Exception as e:
        logger.error(
            f"Copy failed: {source_bucket}/{object_key} → {target_bucket}/{object_key} | {e}"
        )


In [ ]:
SOURCE_BUCKETS = [
    "thienhoang-law",
    "thuvienpl-biz",
    "thuvienpl.th.crawl",
    "v03-migrate-v0"
]

TARGET_BUCKET = "v03-object-v1"
MAX_WORKERS = 8   # khuyến nghị 5–10


def merge_buckets_multithread():
    client = init_minio_client()

    # Ensure target bucket exists
    ensure_bucket(client, TARGET_BUCKET)

    tasks = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        for bucket in SOURCE_BUCKETS:
            logger.info(f"Listing objects from bucket: {bucket}")

            for key in list_all_objects(client, bucket):
                tasks.append(
                    executor.submit(
                        copy_object,
                        client,
                        bucket,
                        key,
                        TARGET_BUCKET,
                    )
                )

        for future in as_completed(tasks):
            future.result()

    logger.success("✅ MERGE BUCKETS COMPLETED")


# ===== RUN =====
merge_buckets_multithread()
